In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score


In [4]:
df =pd.read_csv('../day_28/train.csv',usecols=['Age','Fare','Survived'])
df.sample(10)

,Survived,Age,Fare
514,0,24.0,7.4958
205,0,2.0,10.4625
236,0,44.0,26.0000
888,0,NaN,23.4500
421,0,21.0,7.7333
31,1,NaN,146.5208
342,0,28.0,13.0000
242,0,29.0,10.5000
497,0,NaN,15.1000
605,0,36.0,15.5500


In [5]:
df.dropna(inplace=True)
df.shape

(714, 3)

In [6]:
x=df.drop(columns=['Survived'])
y=df['Survived']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)
x_train.shape

(571, 2)

In [11]:
dt=DecisionTreeClassifier()
dt.fit(x_train,y_train)
pred=dt.predict(x_test)
acc=accuracy_score(y_test,pred)
acc


0.6223776223776224

In [12]:
np.mean(cross_val_score(dt,x_train,y_train,cv=10,scoring='accuracy'))

np.float64(0.6010284331518452)

In [13]:
kbin_age=KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')
kbin_fare=KBinsDiscretizer(n_bins=15,encode='ordinal',strategy='quantile')

In [15]:
cltf=ColumnTransformer([('kbin_age',kbin_age,['Age'])
                        ,('kbin_fare',kbin_fare,['Fare'])],remainder='passthrough')

In [16]:
x_train_tran=cltf.fit_transform(x_train)
x_test_trans=cltf.transform(x_test)

c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [18]:
cltf.named_transformers_['kbin_age'].bin_edges_

array([array([ 0.67,  8.  , 16.  , 18.  , 21.  , 23.  , 25.  , 27.  , 29.  ,
              31.  , 34.  , 37.  , 42.  , 47.  , 54.  , 80.  ])             ],
      dtype=object)

In [19]:
output = pd.DataFrame({
    'age':x_train['Age'],
    'age_trf':x_train_tran[:,0],
    'fare':x_train['Fare'],
    'fare_trf':x_train_tran[:,1]
})

In [22]:
output['age_labels'] = pd.cut(x=x_train['Age'],
                                    bins=cltf.named_transformers_['kbin_age'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=x_train['Fare'],
                                    bins=cltf.named_transformers_['kbin_fare'].bin_edges_[0].tolist())

In [23]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
272,41.0,11.0,19.5000,8.0,"(37.0, 42.0]","(18.0, 25.929]"
370,25.0,6.0,55.4417,12.0,"(23.0, 25.0]","(39.688, 57.979]"
521,22.0,4.0,7.8958,3.0,"(21.0, 23.0]","(7.775, 7.896]"
703,25.0,6.0,7.7417,1.0,"(23.0, 25.0]","(7.229, 7.775]"
687,19.0,3.0,10.1708,5.0,"(18.0, 21.0]","(9.588, 12.35]"


In [24]:
clf = DecisionTreeClassifier()
clf.fit(x_train_tran,y_train)
y_pred2 = clf.predict(x_test_trans)

In [25]:
accuracy_score(y_test,y_pred2)

0.6573426573426573

In [27]:
X_trf = cltf.fit_transform(x)
np.mean(cross_val_score(DecisionTreeClassifier(),x,y,cv=10,scoring='accuracy'))

c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\mahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


np.float64(0.6330985915492957)

In [ ]:
def discretize(bins,strategy):
    kbin_age = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
    kbin_fare = KBinsDiscretizer(n_bins=bins,encode='ordinal',strategy=strategy)
    
    trf = ColumnTransformer([
        ('first',kbin_age,[0]),
        ('second',kbin_fare,[1])
    ])
    
    X_trf = trf.fit_transform(X)
    print(np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy')))
    
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Age'])
    plt.title("Before")

    plt.subplot(122)
    plt.hist(X_trf[:,0],color='red')
    plt.title("After")

    plt.show()
    
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    plt.hist(X['Fare'])
    plt.title("Before")

    plt.subplot(122)
    plt.hist(X_trf[:,1],color='red')
    plt.title("Fare")

    plt.show()
    
    discretize(5,'kmeans')